<a href="https://colab.research.google.com/github/fleurclochette/Scientific-Project-SS26/blob/main/notebooks/01_data_collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import os
import subprocess
import numpy as np
import pandas as pd
import yfinance as yf
from google.colab import userdata

# Securely fetching credentials from Colab Secrets
GH_TOKEN = userdata.get('GITHUB_TOKEN')
GH_USER = userdata.get('GITHUB_USER')
GH_EMAIL = "tushar.latiyan@st.ovgu.de"
REPO_URL = f"https://{GH_TOKEN}@github.com/fleurclochette/Scientific-Project-SS26.git"

# Defining the exact 23-asset universe
tickers = [
    '^GSPC', 'XLB', 'XLE', 'XLF', 'XLI', 'XLK', 'XLP', 'XLU', 'XLV', 'XLY',
    '^AXJO', '^DJI', '^FCHI', '^FTSE', '^GDAXI', '^GSPTSE', '^HSI',
    '^IXIC', '^KS11', '^N225', '^RUT', '^STOXX50E', '^TWII'
]

print("Pulling market data from Jan 2000 through July 2026...")

# Downloading data (auto_adjust=False prevents the KeyError)
# end='2026-08-01' is exclusive, ensuring the final day captured is July 31, 2026.
df_raw = yf.download(tickers, start='2000-01-01', end='2026-08-01', auto_adjust=False)['Adj Close']

# Cleaning and calculating
# Forward filling to sync global trading holidays, then dropping initial NaNs
prices = df_raw.ffill().dropna()

# Calculating daily log returns for Karin's EGARCH models
log_returns = np.log(prices / prices.shift(1)).dropna()

# CROSS-CHECK: Verifying the time frame explicitly before saving
start_date = prices.index.min().date()
end_date = prices.index.max().date()
print(f"\n--- DATA VALIDATION ---")
print(f"First Trading Day: {start_date}")
print(f"Last Trading Day:  {end_date}")
if str(end_date) == '2026-07-31':
    print("Timeframe check passed: Data ends exactly on the last trading day of July 2026.")
else:
    print(f"WARNING: End date is {end_date}, not July 31, 2026.")
print(f"Total Rows (Prices): {len(prices)}")
print(f"Total Rows (Returns): {len(log_returns)}")
print(f"-----------------------\n")

# Saving both files locally
os.makedirs('data/raw', exist_ok=True)
prices_path = 'data/raw/master_23_assets_prices_2000_2026.csv'
returns_path = 'data/raw/master_23_assets_returns_2000_2026.csv'

prices.to_csv(prices_path)
log_returns.to_csv(returns_path)
print("Saved both raw prices and log returns locally.")

# Pshing both files securely to the main branch
print("Pushing both files securely to GitHub...")
cmds = [
    "rm -rf temp_repo",
    f"git clone {REPO_URL} temp_repo",
    "mkdir -p temp_repo/data/raw",
    f"cp {prices_path} temp_repo/data/raw/",
    f"cp {returns_path} temp_repo/data/raw/",
    f'cd temp_repo && git config user.email "{GH_EMAIL}" && git config user.name "{GH_USER}"',
    'cd temp_repo && git add data/raw/master_23_assets_prices_2000_2026.csv data/raw/master_23_assets_returns_2000_2026.csv',
    'cd temp_repo && git commit -m "chore: add both raw prices and log returns (Jan 2000 - Jul 2026)"',
    'cd temp_repo && git push origin main',
    "rm -rf temp_repo"
]

for cmd in cmds:
    # We suppress standard output so it doesn't print your token if git complains
    subprocess.run(cmd, shell=True, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("Git push complete. Both files are locked into the repository.")

Pulling market data from Jan 2000 through July 2026...


[*********************100%***********************]  23 of 23 completed



--- DATA VALIDATION ---
First Trading Day: 2007-03-30
Last Trading Day:  2026-07-31
Timeframe check passed: Data ends exactly on the last trading day of July 2026.
Total Rows (Prices): 5032
Total Rows (Returns): 5031
-----------------------

Saved both raw prices and log returns locally.
Pushing both files securely to GitHub...
Git push complete. Both files are locked into the repository.
